# NB14: CPTAC external validation with frozen TCGA-trained scorer

Applies the frozen TCGA-trained PCA -> Ridge -> Platt model from NB07
directly to CPTAC OpenCLIP ViT-B/16 patient embeddings, with no
retraining or hyperparameter tuning.

**Manuscript reference** (Methods, Results, Figure 4B):
- Frozen TCGA-trained scorer (PCA=384, Ridge alpha=30.0, Platt) from NB07
- Backbone: OpenCLIP ViT-B/16 (LAION-2B, 512-d) - same as TCGA training
- HRD threshold for CPTAC: scarHRD >= 42
- CPTAC-LUAD (n=106): AUROC = 0.671 (95% CI: 0.487 - 0.825)
- Bootstrap reps: 2000 (external; Supp Table S5)
- Labels source: Loeffler et al. (2024) BMC Biology

**Required env**: `WORKSPACE`, `CPTAC_OPENCLIP_DIR` (directory containing
`cptac_<cohort>_openclip_patient_embeddings.parquet` per cohort).
**Inputs**: NB07 `models/frozen_model.joblib`; NB13
`labels/cptac_<cohort>_loeffler.csv`.
**Outputs**: `results/cptac_external_openclip/{per_cohort_metrics.csv,
per_cohort_predictions/<cohort>_preds.csv, report.json}`,
`figures/fig4b_cptac_roc_overlay.{png,pdf}`.

In [ ]:
import os
import json
from pathlib import Path

import numpy as np
import pandas as pd
import joblib
from sklearn.metrics import roc_auc_score, average_precision_score, brier_score_loss, roc_curve
import matplotlib.pyplot as plt

# env
WORKSPACE = Path(os.environ.get("WORKSPACE", "./workspace")).resolve()
CPTAC_OPENCLIP_DIR = Path(os.environ["CPTAC_OPENCLIP_DIR"]).resolve()
LABELS_DIR = WORKSPACE / "labels"
MODELS_DIR = WORKSPACE / "models"
FIG_DIR = WORKSPACE / "figures"
RESULTS_DIR = WORKSPACE / "results" / "cptac_external_openclip"
PREDS_DIR = RESULTS_DIR / "per_cohort_predictions"
FIG_DIR.mkdir(parents=True, exist_ok=True)
PREDS_DIR.mkdir(parents=True, exist_ok=True)

SEED = 42
BOOT_N = 2000
HRD_THRESHOLD_CPTAC = 42

COHORTS = {
    "LUAD":  {"manuscript_AUROC": 0.671, "manuscript_CI": [0.487, 0.825],
              "manuscript_n": 106, "report_in_fig4b": True},
    "LUSC":  {"manuscript_AUROC": None, "report_in_fig4b": True},
    "HNSCC": {"manuscript_AUROC": None, "report_in_fig4b": True},
    "UCEC":  {"manuscript_AUROC": None, "report_in_fig4b": False},
}

print(f"CPTAC_OPENCLIP_DIR : {CPTAC_OPENCLIP_DIR}")
print(f"frozen model       : {MODELS_DIR / 'frozen_model.joblib'}")
print(f"BOOT_N (external)  : {BOOT_N}")
print(f"HRD threshold      : {HRD_THRESHOLD_CPTAC}")


In [ ]:
frozen_path = MODELS_DIR / "frozen_model.joblib"
assert frozen_path.exists(), f"missing {frozen_path}; run NB07 first"
frozen = joblib.load(frozen_path)
pipe = frozen["pipe"]
platt = frozen["platt"]
exp_dim = int(frozen["feat_dim"])
print(f"frozen pipe: {pipe.named_steps}")
print(f"expected feature dim: {exp_dim}")

def predict_prob(X_arr):
    z = pipe.predict(X_arr).reshape(-1, 1)
    return platt.predict_proba(z)[:, 1]


In [ ]:
def boot_ci(y, p, fn, n_boot=BOOT_N, seed=SEED):
    rng = np.random.default_rng(seed)
    n = len(y)
    vals = []
    for _ in range(n_boot):
        idx = rng.integers(0, n, size=n)
        yb, pb = y[idx], p[idx]
        if yb.sum() == 0 or yb.sum() == n:
            continue
        try:
            vals.append(fn(yb, pb))
        except Exception:
            pass
    if not vals:
        return (float("nan"), float("nan"))
    return float(np.percentile(vals, 2.5)), float(np.percentile(vals, 97.5))

def score_cohort(cohort):
    feats_path = CPTAC_OPENCLIP_DIR / f"cptac_{cohort.lower()}_openclip_patient_embeddings.parquet"
    labels_path = LABELS_DIR / f"cptac_{cohort.lower()}_loeffler.csv"

    if not feats_path.exists():
        return {"cohort": cohort, "error": f"missing features: {feats_path}"}
    if not labels_path.exists():
        return {"cohort": cohort, "error": f"missing labels: {labels_path}; run NB13"}

    X = pd.read_parquet(feats_path)
    if "patient" in X.columns:
        X = X.set_index("patient")
    X.index = X.index.astype(str).str.upper().str.strip()

    L = pd.read_csv(labels_path)
    L["patient"] = L["patient"].astype(str).str.upper().str.strip()
    L = L.dropna(subset=["HRD_binary"]).drop_duplicates("patient").set_index("patient")

    common = sorted(set(X.index) & set(L.index))
    X = X.loc[common]
    L = L.loc[common]

    if X.shape[1] != exp_dim:
        return {"cohort": cohort, "error": (f"feature dim {X.shape[1]} != frozen model expects {exp_dim}")}

    y = L["HRD_binary"].astype(int).values
    if y.sum() == 0 or y.sum() == len(y):
        return {"cohort": cohort, "error": f"single-class labels (n_pos={int(y.sum())} of {len(y)})"}

    p = predict_prob(X.values.astype(np.float32))
    auc = float(roc_auc_score(y, p))
    ap = float(average_precision_score(y, p))
    br = float(brier_score_loss(y, p))
    auc_lo, auc_hi = boot_ci(y, p, roc_auc_score)
    ap_lo, ap_hi = boot_ci(y, p, average_precision_score)

    preds_df = pd.DataFrame({"patient": common, "y_true": y, "y_score": p,
                             "HRD_score": L["HRD_score"].values})
    preds_df.to_csv(PREDS_DIR / f"{cohort.lower()}_preds.csv", index=False)

    return {
        "cohort": cohort, "n": int(len(y)),
        "n_pos": int(y.sum()), "n_neg": int(len(y) - y.sum()),
        "AUROC": auc, "AUROC_lo": auc_lo, "AUROC_hi": auc_hi,
        "AP": ap, "AP_lo": ap_lo, "AP_hi": ap_hi, "Brier": br,
    }


In [ ]:
rows = []
for cohort, cfg in COHORTS.items():
    print(f"\n--- CPTAC-{cohort} ---")
    res = score_cohort(cohort)
    res.update({"manuscript_AUROC": cfg.get("manuscript_AUROC"),
                "manuscript_CI": cfg.get("manuscript_CI"),
                "manuscript_n": cfg.get("manuscript_n")})
    if "error" in res:
        print(f"  {res['error']}")
    else:
        ms = cfg.get("manuscript_AUROC")
        ms_str = f" (ms ref {ms:.3f})" if ms else ""
        print(f"  n={res['n']}  HRD+={res['n_pos']}  HRD-={res['n_neg']}")
        print(f"  AUROC = {res['AUROC']:.3f} ({res['AUROC_lo']:.3f}-{res['AUROC_hi']:.3f}){ms_str}")
        print(f"  AP    = {res['AP']:.3f}  Brier = {res['Brier']:.3f}")
    rows.append(res)

df = pd.DataFrame(rows)
df.to_csv(RESULTS_DIR / "per_cohort_metrics.csv", index=False)
print()
print(df.to_string(index=False))


In [ ]:
fig, ax = plt.subplots(figsize=(5.0, 4.5))
colors = {"LUAD": "C3", "LUSC": "C0", "HNSCC": "C2", "UCEC": "C1"}

n_drawn = 0
for cohort, cfg in COHORTS.items():
    if not cfg.get("report_in_fig4b", False):
        continue
    preds_path = PREDS_DIR / f"{cohort.lower()}_preds.csv"
    if not preds_path.exists():
        continue
    pdf = pd.read_csv(preds_path)
    y, p = pdf["y_true"].values, pdf["y_score"].values
    if y.sum() == 0 or y.sum() == len(y):
        continue
    fpr, tpr, _ = roc_curve(y, p)
    auc = roc_auc_score(y, p)
    ax.plot(fpr, tpr, color=colors.get(cohort, "C0"), lw=1.8,
            label=f"CPTAC-{cohort} (n={len(y)}, AUROC = {auc:.3f})")
    n_drawn += 1

ax.plot([0, 1], [0, 1], color="0.6", lw=0.8, ls=":")
ax.set_xlim(0, 1); ax.set_ylim(0, 1.005)
ax.set_xlabel("False positive rate")
ax.set_ylabel("True positive rate")
ax.set_title("Figure 4B: frozen TCGA-trained scorer on CPTAC cohorts")
ax.legend(loc="lower right", fontsize=8, frameon=False)
fig.tight_layout()
fig.savefig(FIG_DIR / "fig4b_cptac_roc_overlay.png", dpi=300)
fig.savefig(FIG_DIR / "fig4b_cptac_roc_overlay.pdf")
plt.show()
print(f"\ncohorts plotted: {n_drawn}")


In [ ]:
report = {
    "seed": SEED,
    "bootstrap_n": BOOT_N,
    "hrd_threshold_cptac": HRD_THRESHOLD_CPTAC,
    "frozen_model_path": str(frozen_path),
    "results": rows,
    "manuscript_refs": {
        "CPTAC_LUAD_AUROC": 0.671,
        "CPTAC_LUAD_CI": [0.487, 0.825],
        "CPTAC_LUAD_n": 106,
        "label_source": "Loeffler et al. 2024 BMC Biology",
    },
}
(RESULTS_DIR / "report.json").write_text(json.dumps(report, indent=2, default=str))
print(json.dumps(report, indent=2, default=str))
